In [18]:
%load_ext autoreload
%autoreload 2

from parser_functions import PlanParser, DocumentParser
doc1_path = r"C:\Users\egorg\Documents\RAG_минцифры\Правильно\Заявка в ПГ ЛОШ.docx"
doc2_path = r"C:\Users\egorg\Documents\RAG_минцифры\Правильно\Проект Контракта ЛОШ.docx"


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [77]:
doc_parcer = DocumentParser(doc2_path)
table_data = doc_parcer.table_to_markdown()
print(table_data[:10000])

| «___»_______20___ г. | г. Новосибирск |
| --- | --- |


| Наименование товара, работы, услуги | Наименование характеристики | Значение характеристики | Единица измерения характеристики | Количество, штук |
| --- | --- | --- | --- | --- |
| Ноутбук | Размер диагонали экрана | ≥ 15.6 | Дюйм (25,4 мм) | 34 |
| Ноутбук | Разрешение экрана | Full HD |  | 34 |
| Ноутбук | Технология изготовления матрицы дисплея | IPS (PLS, ADS, AAS, FFS, SFT, New Mode2, Vistarich) |  | 34 |
| Ноутбук | Яркость экрана, кд/м2 | ≥ 300 |  | 34 |
| Ноутбук | Количество ядер процессора | ≥ 6 | Штука | 34 |
| Ноутбук | Количество потоков процессора | ≥ 12 | Штука | 34 |
| Ноутбук | Частота процессора базовая | ≥ 2.3 | Гигагерц | 34 |
| Ноутбук | Тип видеоадаптера | Интегрированная (встроенная) |  | 34 |
| Ноутбук | Тип оперативной памяти | DDR4 |  | 34 |
| Ноутбук | Общий объем установленной оперативной памяти | ≥ 32 | Гигабайт | 34 |
| Ноутбук | Максимальный общий поддерживаемый объем оперативной памяти | ≥ 32 |

In [16]:
plain_text = doc_parcer.extract_clean_text()
print(plain_text[:2000])

Контракт №

на поставку машин

Идентификационный код закупки указывается в структурированном виде (цифровой форме) электронного контракта, сформированного с использованием единой информационной системы в сфере закупок.

Государственное бюджетное учреждение Новосибирской области "Центр информационных технологий Новосибирской области" (сокращенное наименование - ГБУ НСО "ЦИТ НСО"), именуемый в дальнейшем "Заказчик" в лице руководителя учреждения Брумма Сергея Викторовича, действующего на основании Устава, с одной стороны, и _______________________,_______________________________(сокращенное наименование - __________) именуем___ в дальнейшем "Поставщик", в лице ________________________, действующ___ на основании _______________________, с другой стороны, вместе именуемые в дальнейшем "Стороны", в соответствии с частью 12 статьи 93 Федерального закона от 05.04.2013 № 44-ФЗ "О контрактной системе в сфере закупок товаров, работ, услуг для обеспечения государственных и муниципальных нужд", на

In [4]:

parser_plan = PlanParser(doc1_path)
plan_points = parser_plan.extract_table_kv_from_docx()
plan_points
key_words = [
    "Наименование объекта закупки",
    "Код позиции КТРУ",
    "Количество",
    "Сроки поставки товара",
    "Место поставки товара"
    ]
plan_points_use = [
    plan_point
    for plan_point in plan_points
    if any(kw.lower() in plan_point.lower() for kw in key_words)
]
if not plan_points_use:
    raise ValueError(f"Не найдено пунктов в план-графике, соответствующих ключевым словам, \n plan_points:\n{plan_points}")

plan_points_use

['Наименование объекта закупки: Поставка носков',
 'Код позиции КТРУ: Код позиции: ; 26.20.11.110-10000155 - Носок',
 'Количество: Носки - 34 шт.',
 'Сроки поставки товара, выполнения работ, оказания услуг по контракту: В течение 48 (сорока восьми) календарных дней с даты заключения Контракта.',
 'Место поставки товара, оказания услуг, выполнения работ: Российская Федерация, Новосибирская область, г.о. город Новосибирск, г Новосибирск, пер Архонский, д. 10.']

In [24]:
import sys
sys.path.insert(0, r"C:\Users\egorg\Documents\RAG_минцифры\git_repo_clone\Documents-verification-Mintsifry")


In [74]:
from new_model.parser_functions import PlanParser, DocumentParser
from new_model.retriever import Retriever
from new_model.embeddings import get_embeddings


parser_contract = DocumentParser(doc2_path)
paragraphs_contract = parser_contract.extract_clean_text()
tables_contract = parser_contract.table_to_markdown()
contract_full_text = ("Название: " + paragraphs_contract + "\n\n" + tables_contract).strip()

if not contract_full_text:
    raise ValueError("Не удалось извлечь данные из контракта: contract_full_text пуст")

parser_plan = PlanParser(doc1_path)
plan_points = parser_plan.extract_table_kv_from_docx()
if not plan_points:
    raise ValueError("Не удалось извлечь данные из плана-графика: plan_points пуст")

key_words = [
    # "Наименование объекта закупки",
    # "Код позиции КТРУ",
    # "Количество",
    "Сроки поставки товара",
    "Место поставки товара"
    ]

# оставлю только адекватные
plan_points = [
    plan_point
    for plan_point in plan_points
    if any(kw.lower() in plan_point.lower() for kw in key_words)
]
if not plan_points:
    raise ValueError("После фильтрации не осталось подходящих пунктов плана-графика")

faiss = Retriever(embeddings=get_embeddings())
retriever = faiss.create_retriever(texts=[contract_full_text], n=20)

In [72]:
# print(paragraphs_contract[-4000:-2000])

In [67]:
import re

STOP_PHRASES = [
    "по контракту",
    "выполнения работ",
    "оказания услуг",
    "товара",
    "объекта закупки",
    "позиции",
]

def build_retrieval_query(plan_point: str) -> str:
    text = plan_point.strip().lower()

    if ":" in text:
        left, right = text.split(":", 1)
        text = right.strip() or text

    for phrase in STOP_PHRASES:
        text = text.replace(phrase, " ")

    text = re.sub(r"[^\w\s\-.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [t for t in tokens if len(t) > 2 or any(ch.isdigit() for ch in t)]

    return " ".join(tokens[:12])

In [69]:
build_retrieval_query(plan_points[0])

'течение 48 сорока восьми календарных дней даты заключения контракта.'

In [76]:

# for i, plan_point in enumerate(plan_points, start=1):
build_retrieval_query(plan_points[0])

# plan_point = ' В течение 48 (сорока восьми) календарных дней.'
plan_point = plan_points[0]
docs = retriever.invoke(plan_point)
print("=" * 80)
print(f"plan_point {1}: {plan_point}")
print("-" * 80)

for j, doc in enumerate(docs, start=1):
    print(f"chunk {j}:")
    print(doc.page_content[:1500])
    print("-" * 80)


plan_point 1: Сроки поставки товара, выполнения работ, оказания услуг по контракту: В течение 48 (сорока восьми) календарных дней с даты заключения Контракта.
--------------------------------------------------------------------------------
chunk 1:
системы в сфере закупок (далее - место доставки) в срок: в течение 48 (сорока трёх) календарных дней с даты заключения Контракта.
--------------------------------------------------------------------------------
chunk 2:
в соответствии с Федеральным законом от 5 апреля 2013 г. № 44-ФЗ "О контрактной системе в сфере закупок товаров, работ, услуг для обеспечения государственных и муниципальных нужд" в порядке и в сроки, установленные разделом VIII Контракта.
--------------------------------------------------------------------------------
chunk 3:
3.1. Поставщик самостоятельно доставляет Товар Заказчику по адресу, указанному в структурированном виде (цифровой форме) электронного контракта, сформированного с использованием единой информационной с